In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print("Libraries imported successfully")

Libraries imported successfully


In [3]:
df = pd.read_csv("sportsdata_set")

print("Dataset loaded successfully")
display(df.head())

FileNotFoundError: [Errno 2] No such file or directory: 'sportsdata_set'

In [ ]:
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])
print("Dataset shape:", df.shape)

In [ ]:
print("Column names:")
print(df.columns.tolist())

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
print("Missing values in each column:")

print(df.isnull().sum())

In [ ]:
missing_percentage = (df.isnull().sum() / len(df)) * 100

print(missing_percentage)

In [ ]:
missing_report = pd.DataFrame({
    "Missing Count": df.isnull().sum(),
    "Missing Percentage": (
        df.isnull().sum() / len(df) * 100
    ).round(2)
})

display(missing_report)

In [ ]:
plt.figure(figsize=(10, 5))

sns.heatmap(df.isnull(), cbar=False)

plt.title("Missing Values Heatmap")
plt.xlabel("Columns")
plt.ylabel("Rows")

plt.show()

In [ ]:
numeric_columns = df.select_dtypes(
    include=np.number
).columns

categorical_columns = df.select_dtypes(
    exclude=np.number
).columns

# Fill numeric missing values with median
for col in numeric_columns:
    df[col] = df[col].fillna(df[col].median())

# Fill categorical missing values with mode
for col in categorical_columns:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].mode()[0])

print("Missing values handled")
print(df.isnull().sum())

In [ ]:
print("Total duplicate rows:", df.duplicated().sum())

In [ ]:
duplicates = df[df.duplicated(keep=False)]

display(duplicates)

In [ ]:
print("Rows before:", len(df))

df = df.drop_duplicates()

print("Rows after:", len(df))

In [ ]:
print("Remaining duplicates:", df.duplicated().sum())

In [ ]:
print(df["Position"].unique())

In [ ]:
print(df["Position"].value_counts())

In [ ]:
numeric_columns = ["Age", "Overall", "Potential", "Finishing"]

for col in numeric_columns:
    print(f"\n{col} zero count:", (df[col] == 0).sum())

In [ ]:
for col in ["Age", "Overall", "Potential", "Finishing"]:
    print(f"Negative {col}:", (df[col] < 0).sum())

In [ ]:
print("Preferred Foot values:")
print(df["Preferred Foot"].value_counts())

In [ ]:
plt.figure(figsize=(8, 5))

sns.boxplot(y=df["Overall"])

plt.title("Boxplot of Overall Rating")

plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

sns.boxplot(y=df["Finishing"])

plt.title("Boxplot of Finishing")

plt.show()

In [ ]:
def detect_outliers(column):

    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)

    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = df[
        (df[column] < lower_bound) |
        (df[column] > upper_bound)
    ]

    print("Column:", column)
    print("Q1:", Q1)
    print("Q3:", Q3)
    print("IQR:", IQR)
    print("Outliers:", len(outliers))

    return outliers

overall_outliers = detect_outliers("Overall")
finishing_outliers = detect_outliers("Finishing")

In [ ]:
plt.figure(figsize=(8, 5))

sns.histplot(df["Overall"], kde=True)

plt.title("Distribution of Overall Rating")
plt.xlabel("Overall")
plt.ylabel("Frequency")

plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

sns.histplot(df["Finishing"], kde=True)

plt.title("Distribution of Finishing")
plt.xlabel("Finishing")
plt.ylabel("Frequency")

plt.show()

In [ ]:
print("Overall skewness:", df["Overall"].skew())

print("Finishing skewness:", df["Finishing"].skew())

In [ ]:
position_counts = df["Position"].value_counts()

display(position_counts)

In [ ]:
plt.figure(figsize=(12, 5))

sns.countplot(
    data=df,
    x="Position",
    order=df["Position"].value_counts().index
)

plt.title("Number of Players by Position")
plt.xlabel("Position")
plt.ylabel("Count")
plt.xticks(rotation=90)

plt.show()

In [ ]:
# Group the smaller positions into "Other" so the pie chart stays readable
top_positions = df["Position"].value_counts().head(8)
other_count = df["Position"].value_counts().iloc[8:].sum()

pie_data = top_positions.copy()
pie_data["Other"] = other_count

pie_data.plot(
    kind="pie",
    autopct="%1.1f%%",
    figsize=(7, 7)
)

plt.title("Position Distribution (Top 8 + Other)")
plt.ylabel("")

plt.show()

In [ ]:
print("Average Overall rating:", df["Overall"].mean())

print("Median Overall rating:", df["Overall"].median())

print("Maximum Overall rating:", df["Overall"].max())

print("Minimum Overall rating:", df["Overall"].min())

In [ ]:
overall_by_position = df.groupby(
    "Position"
)["Overall"].agg([
    "mean",
    "min",
    "max",
    "count"
])

display(overall_by_position.sort_values("mean", ascending=False))

In [ ]:
df.groupby("Position")["Overall"].mean().sort_values(ascending=False).plot(
    kind="bar",
    figsize=(12, 5)
)

plt.title("Average Overall Rating by Position")
plt.xlabel("Position")
plt.ylabel("Average Overall")

plt.xticks(rotation=90)

plt.show()

### Derived target: High Potential

The dataset has no ready-made success/fail label, so one is created: a player is flagged as **High Potential** (1) when their `Potential` rating is higher than their current `Overall` rating — i.e. they still have room to grow. Otherwise they're flagged as **At Ceiling** (0).

In [ ]:
df["High Potential"] = (
    df["Potential"] > df["Overall"]
).astype(int)

display(df[["Name", "Overall", "Potential", "High Potential"]].head())

In [ ]:
high_potential_count = (df["High Potential"] == 1).sum()

at_ceiling_count = (df["High Potential"] == 0).sum()

total_count = len(df)

print("High Potential:", high_potential_count)

print("At Ceiling:", at_ceiling_count)

print("Total:", total_count)

In [ ]:
high_potential_percentage = (
    high_potential_count / total_count
) * 100

print(f"High Potential Percentage: {high_potential_percentage:.2f}%")

In [ ]:
at_ceiling_percentage = (
    at_ceiling_count / total_count
) * 100

print(f"At Ceiling Percentage: {at_ceiling_percentage:.2f}%")

In [ ]:
potential_analysis = pd.DataFrame({
    "Category": [
        "High Potential",
        "At Ceiling",
        "Total"
    ],

    "Count": [
        high_potential_count,
        at_ceiling_count,
        total_count
    ],

    "Percentage": [
        high_potential_percentage,
        at_ceiling_percentage,
        100.00
    ]
})

potential_analysis["Percentage"] = (
    potential_analysis["Percentage"].round(2)
)

display(potential_analysis)

In [ ]:
plt.figure(figsize=(8, 5))

plt.bar(
    ["High Potential", "At Ceiling"],
    [high_potential_count, at_ceiling_count]
)

plt.title("High Potential vs At Ceiling Players")
plt.xlabel("Status")
plt.ylabel("Number of Players")

plt.show()

In [ ]:
plt.figure(figsize=(7, 7))

plt.pie(
    [high_potential_count, at_ceiling_count],
    labels=["High Potential", "At Ceiling"],
    autopct="%1.1f%%"
)

plt.title("High Potential Percentage")

plt.show()

In [ ]:
potential_by_position = pd.crosstab(
    df["Position"],
    df["High Potential"]
)

potential_by_position.columns = [
    "At Ceiling",
    "High Potential"
]

display(potential_by_position)

In [ ]:
potential_rate = df.groupby(
    "Position"
)["High Potential"].mean() * 100

potential_rate = potential_rate.round(2)

display(potential_rate.sort_values(ascending=False))

In [ ]:
plt.figure(figsize=(12, 5))

potential_rate.sort_values(ascending=False).plot(kind="bar")

plt.title("High Potential Rate by Position")
plt.xlabel("Position")
plt.ylabel("High Potential Rate (%)")

plt.xticks(rotation=90)

plt.show()

In [ ]:
print("Average Finishing:", df["Finishing"].mean())

print("Median Finishing:", df["Finishing"].median())

print("Maximum Finishing:", df["Finishing"].max())

print("Minimum Finishing:", df["Finishing"].min())

In [ ]:
finishing_by_position = df.groupby(
    "Position"
)["Finishing"].agg([
    "mean",
    "min",
    "max"
])

display(finishing_by_position.sort_values("mean", ascending=False))

In [ ]:
df.groupby("Position")["Finishing"].mean().sort_values(ascending=False).plot(
    kind="bar",
    figsize=(12, 5)
)

plt.title("Average Finishing by Position")
plt.xlabel("Position")
plt.ylabel("Average Finishing")

plt.xticks(rotation=90)

plt.show()

In [ ]:
correlation = df[
    ["Age", "Overall", "Potential", "Finishing"]
].corr()

display(correlation)

In [ ]:
plt.figure(figsize=(8, 6))

sns.heatmap(
    correlation,
    annot=True,
    cmap="coolwarm",
    fmt=".2f"
)

plt.title("Correlation Heatmap")

plt.show()

In [ ]:
print("Final dataset shape:", df.shape)

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())

print("\nData types:")
print(df.dtypes)

In [ ]:
display(df.head(10))

In [ ]:
output_file = "sportsdata_cleaned_final.csv"

df.to_csv(
    output_file,
    index=False
)

print("Cleaned dataset saved successfully")

### Conclusion

The player-ratings sports dataset was analyzed using Python, Pandas, and Seaborn. Data profiling, missing-value analysis, duplicate detection, data validation, IQR-based outlier detection, statistical analysis, and visualization were performed. Player ratings, positions, and a derived High Potential flag (Potential higher than current Overall) were analyzed to understand squad composition and identify players with room left to grow.